# Long-horizon forecasting and input-noise robustness

This notebook is a cleaned record of the original training program. Execution outputs and notebook execution metadata were removed; computational cells were preserved to maintain the historical experiment protocol. Dataset access and optional cache paths may require configuration.


In [ ]:

# ============================================================
# 0. Runtime
# ============================================================
import os, sys, math, time, json, hashlib, shutil, subprocess, zipfile, copy, gc
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import h5py
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda"

ROOT=Path(os.getenv("CIDM_BENCH_ROOT",str(Path.cwd())))
OUT=ROOT/"CIDM_Foundation_A4_R1_LongRollout_RESULTS"
SRC=OUT/"sources"; FROZEN=OUT/"frozen"; CKPT=OUT/"checkpoints"
CACHE=OUT/"restored_A2_cache"; LONGCACHE=OUT/"long_eval_cache"
FIG=OUT/"figures"; FIGDATA=OUT/"figure_data"
RAW=Path(os.getenv("CIDM_PDEBENCH_DATA_ROOT","/content/CIDM_PDEBench_Data"))
for p in [OUT,SRC,FROZEN,CKPT,CACHE,LONGCACHE,FIG,FIGDATA,RAW]:
    p.mkdir(parents=True,exist_ok=True)

HISTORY=4
TRAIN_HORIZON=6
SEEDS=[17,41,73]
MAX_UPDATES=int(os.getenv("CIDM_A4_R1_MAX_UPDATES","2500"))
MIN_UPDATES=int(os.getenv("CIDM_A4_R1_MIN_UPDATES","1000"))
EVAL_EVERY=100
PATIENCE_EVALS=6
REL_MIN_DELTA=0.001
WEIGHT_DECAY=1e-4
BATCH_1D=16
BATCH_2D=4
REQUESTED_HORIZONS=[6,12,24,48]

TASKS={
    "advection_1d":{
        "family":"transport","dim":1,"channels":1,"spatial":[256],
        "filename":"1D_Advection_Sols_beta0.4.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/255674",
        "md5":"d595bbfd2c659df995a93cd40d6ea568",
        "stride":4,
    },
    "burgers_1d":{
        "family":"nonlinear_transport","dim":1,"channels":1,"spatial":[256],
        "filename":"1D_Burgers_Sols_Nu0.01.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/281363",
        "md5":"e6d9a4f62baf9a29121a816b919e2770",
        "stride":4,
    },
    "reacdiff_2d":{
        "family":"reaction_diffusion_2d","dim":2,"channels":2,"spatial":[128,128],
        "filename":"2D_diff-react_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133017",
        "md5":"b8d0b86064193195ddc30c33be5dc949",
        "stride":1,
    },
    "swe_2d":{
        "family":"shallow_water","dim":2,"channels":1,"spatial":[128,128],
        "filename":"2D_rdb_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133021",
        "md5":"75d838c47aa410694bdc912ea7f22282",
        "stride":1,
    },
}

MODELS=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
]
LR={
    "CIDM-Foundation-v1":1e-4,
    "CIDM-Complete":1e-3,
    "PDEBench-FNO":3e-3,
    "NeuralOperator-FNO":3e-3,
}
CONFIG={
    "advection_1d":{
        "CIDM-Foundation-v1":{"latent_dim":104,"hidden":160},
        "CIDM-Complete":{"latent_dim":42,"hidden":72,"soc_modes1":1,"soc_modes2":4},
        "PDEBench-FNO":{"width":70,"modes":24},
        "NeuralOperator-FNO":{"width":88,"modes":24},
    },
    "burgers_1d":{
        "CIDM-Foundation-v1":{"latent_dim":104,"hidden":160},
        "CIDM-Complete":{"latent_dim":42,"hidden":72,"soc_modes1":1,"soc_modes2":4},
        "PDEBench-FNO":{"width":70,"modes":24},
        "NeuralOperator-FNO":{"width":88,"modes":24},
    },
    "reacdiff_2d":{
        "CIDM-Foundation-v1":{"latent_dim":64,"hidden":96},
        "CIDM-Complete":{"latent_dim":20,"hidden":92,"soc_modes1":2,"soc_modes2":14},
        "PDEBench-FNO":{"width":18,"modes":14},
        "NeuralOperator-FNO":{"width":68,"modes":6},
    },
    "swe_2d":{
        "CIDM-Foundation-v1":{"latent_dim":36,"hidden":108},
        "CIDM-Complete":{"latent_dim":22,"hidden":84,"soc_modes1":4,"soc_modes2":12},
        "PDEBench-FNO":{"width":18,"modes":14},
        "NeuralOperator-FNO":{"width":68,"modes":6},
    },
}

REPORT={
    "experiment":"CIDM Foundation A4-R1 Long Rollout & Perturbation Generalization",
    "started_at":time.strftime("%Y-%m-%d %H:%M:%S"),
    "seeds":SEEDS,
    "requested_horizons":REQUESTED_HORIZONS,
}

def run(cmd,cwd=None,timeout=1800):
    p=subprocess.run(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout)
    return p.returncode,p.stdout

def seed_all(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def save_json(x,p):
    Path(p).write_text(json.dumps(x,ensure_ascii=False,indent=2,default=str),encoding="utf-8")

def md5_file(p,chunk=8*1024*1024):
    h=hashlib.md5()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

def sha256_file(p,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

def portable_state_dict(model):
    raw=model.state_dict()
    out=OrderedDict()
    for k,v in raw.items():
        if torch.is_tensor(v): out[k]=v.detach().cpu().clone()
        else: out[k]=copy.deepcopy(v)
    if hasattr(raw,"_metadata"): out._metadata=copy.deepcopy(raw._metadata)
    return out

def load_portable_state_dict(model,sd):
    inc=model.load_state_dict(sd,strict=False)
    miss=list(inc.missing_keys)
    unexp=[k for k in inc.unexpected_keys if k.split(".")[-1]!="_metadata"]
    if miss or unexp: raise RuntimeError((miss,unexp))


## 1. Frozen CIDM Core / Complete audit

In [ ]:

# ============================================================
# 1. Freeze audit:
#    CIDM Foundation Core v1.0 Final
#    + CIDM-Complete archived executable topology
# ============================================================
CIDM_CORE_SOURCE = '"""\nCIDM Foundation Core v1.0\n=========================\n\nFrozen canonical implementation for fair external benchmarking.\n\nThis file contains ONLY the universal core selected after the A1-R1-R3\ncomponent-attribution review:\n\n1. Finite-history predictive information state.\n2. Spatial endpoint encoder.\n3. Temporal-history encoder.\n4. Fusion into a single latent information state.\n5. Bounded local latent residual dynamics.\n6. Bounded positive step size.\n7. Pure latent rollout.\n8. Field decoder.\n\nNot part of the universal core:\n- phase/spectral encoder,\n- skew-symmetric channel rotation,\n- antisymmetric flux,\n- hidden reservoir,\n- causal forcing port,\n- transport branch,\n- cross-scale transfer,\n- BER/SOC,\n- state-sync regularization,\n- Cayley update.\n\nThose are task-specific or training/numerical optionals and must not be silently\nenabled in Foundation Core benchmark runs.\n\nImportant:\nThe numerical caps below are architectural hyperparameters, not physical\nconstants.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass(frozen=True)\nclass CIDMFoundationConfig:\n    spatial_dim: int = 2          # 1 or 2\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n\n    # Frozen benchmark defaults; architectural hyperparameters, not laws.\n    local_cap: float = 0.10\n    step_cap: float = 0.30\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef _conv_nd(dim: int):\n    if dim == 1:\n        return nn.Conv1d\n    if dim == 2:\n        return nn.Conv2d\n    raise ValueError(f"spatial_dim must be 1 or 2, got {dim}")\n\n\nclass ConvBlockND(nn.Module):\n    def __init__(self, dim: int, in_ch: int, out_ch: int, hidden: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, out_ch, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field X_t."""\n\n    def __init__(self, dim: int, channels: int, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, channels, latent_dim, hidden)\n\n    def forward(self, x_t: torch.Tensor) -> torch.Tensor:\n        return self.net(x_t)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode finite history H_t = X_{t-L+1:t}."""\n\n    def __init__(\n        self,\n        dim: int,\n        channels: int,\n        hist_len: int,\n        latent_dim: int,\n        hidden: int,\n    ):\n        super().__init__()\n        self.hist_len = hist_len\n        self.channels = channels\n        self.net = ConvBlockND(\n            dim,\n            channels * hist_len,\n            latent_dim,\n            hidden,\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: [B, L, C, *S]\n        if history.shape[1] != self.hist_len:\n            raise ValueError(\n                f"Expected history length {self.hist_len}, got {history.shape[1]}"\n            )\n        b, l, c, *spatial = history.shape\n        if c != self.channels:\n            raise ValueError(f"Expected {self.channels} channels, got {c}")\n        x = history.reshape(b, l * c, *spatial)\n        return self.net(x)\n\n\nclass InformationStateFusion(nn.Module):\n    """Fuse spatial endpoint and temporal-history views into Psi_t."""\n\n    def __init__(self, dim: int, latent_dim: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(2 * latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            Conv(latent_dim, latent_dim, 1),\n        )\n\n    def forward(\n        self,\n        z_space: torch.Tensor,\n        z_time: torch.Tensor,\n    ) -> torch.Tensor:\n        return self.net(torch.cat([z_space, z_time], dim=1))\n\n\nclass BoundedLocalResidualDynamics(nn.Module):\n    """\n    Minimal universal transition field.\n\n    The residual branch is bounded twice:\n    - branch responsibility: local_cap * sigmoid(local_logit)\n    - global step: step_cap * sigmoid(step_logit)\n\n    This prevents an unconstrained transition from silently dominating the\n    latent information state.\n    """\n\n    def __init__(\n        self,\n        dim: int,\n        latent_dim: int,\n        hidden: int,\n        local_cap: float,\n        step_cap: float,\n    ):\n        super().__init__()\n        self.local_cap = float(local_cap)\n        self.step_cap = float(step_cap)\n\n        self.local_net = ConvBlockND(\n            dim, latent_dim, latent_dim, hidden\n        )\n        self.local_logit = nn.Parameter(torch.tensor(-2.2))\n        self.step_logit = nn.Parameter(torch.tensor(-1.5))\n\n    def vector_field(self, psi: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        local_scale = self.local_cap * torch.sigmoid(self.local_logit)\n        raw = torch.tanh(self.local_net(psi))\n        delta_local = local_scale * raw\n\n        info = {\n            "local_scale": local_scale,\n            "local_strength": delta_local.abs().mean(),\n        }\n        return delta_local, info\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        delta_local, info = self.vector_field(psi)\n        step = self.step_cap * torch.sigmoid(self.step_logit)\n        psi_next = psi + step * delta_local\n\n        info = {\n            **info,\n            "step_size": step,\n            "delta_strength": (step * delta_local).abs().mean(),\n        }\n        return psi_next, info\n\n\nclass FieldDecoder(nn.Module):\n    def __init__(self, dim: int, latent_dim: int, channels: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, latent_dim, channels, hidden)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass CIDMFoundationCore(nn.Module):\n    """\n    Closed Information Dynamics Model — Foundation Core v1.0.\n\n    Input:\n        history: [B, L, C, *S]\n\n    Output dict:\n        pred:       [B, K, C, *S]\n        latent0:    [B, D, *S]\n        latent:     [B, D, *S]\n        latent_seq: [B, K, D, *S]\n        diagnostics\n    """\n\n    VERSION = "CIDM-Foundation-Core-v1.0"\n\n    def __init__(self, config: CIDMFoundationConfig):\n        super().__init__()\n        self.config = config\n\n        d = config.spatial_dim\n        D = config.latent_dim\n        H = config.hidden\n\n        self.space_encoder = SpaceEncoder(\n            d, config.channels, D, H\n        )\n        self.time_encoder = TimeHistoryEncoder(\n            d, config.channels, config.hist_len, D, H\n        )\n        self.fusion = InformationStateFusion(d, D)\n\n        self.dynamics = BoundedLocalResidualDynamics(\n            d, D, H,\n            local_cap=config.local_cap,\n            step_cap=config.step_cap,\n        )\n        self.decoder = FieldDecoder(\n            d, D, config.channels, H\n        )\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        z_space = self.space_encoder(last)\n        z_time = self.time_encoder(history)\n        return self.fusion(z_space, z_time)\n\n    def step(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        return self.dynamics(psi)\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi)\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        K = int(horizon or self.config.horizon)\n\n        psi0 = self.encode(history)\n        psi = psi0\n\n        preds = []\n        states = []\n        local_strengths = []\n        step_sizes = []\n        delta_strengths = []\n\n        spatial_dims = tuple(range(1, psi.ndim))\n        energy0 = (psi ** 2).mean(dim=spatial_dims)\n        energies = []\n\n        for _ in range(K):\n            psi, info = self.step(psi)\n            y = self.decode(psi)\n\n            preds.append(y)\n            states.append(psi)\n            local_strengths.append(info["local_strength"])\n            step_sizes.append(info["step_size"])\n            delta_strengths.append(info["delta_strength"])\n            energies.append((psi ** 2).mean(dim=spatial_dims))\n\n        pred = torch.stack(preds, dim=1)\n        latent_seq = torch.stack(states, dim=1)\n        energy = torch.stack(energies, dim=1)\n\n        diagnostics: Dict[str, torch.Tensor] = {\n            "local_strength": torch.stack(local_strengths).mean(),\n            "step_size": torch.stack(step_sizes).mean(),\n            "delta_strength": torch.stack(delta_strengths).mean(),\n            "latent_energy_drift": (energy[:, -1] - energy0).abs().mean(),\n        }\n\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n\n        return {\n            "pred": pred,\n            "latent0": psi0,\n            "latent": psi,\n            "latent_seq": latent_seq,\n            "diagnostics": diagnostics,\n        }\n\n\ndef build_cidm_foundation_core(\n    *,\n    spatial_dim: int,\n    channels: int,\n    hist_len: int,\n    horizon: int,\n    latent_dim: int,\n    hidden: int,\n    local_cap: float = 0.10,\n    step_cap: float = 0.30,\n) -> CIDMFoundationCore:\n    cfg = CIDMFoundationConfig(\n        spatial_dim=spatial_dim,\n        channels=channels,\n        hist_len=hist_len,\n        horizon=horizon,\n        latent_dim=latent_dim,\n        hidden=hidden,\n        local_cap=local_cap,\n        step_cap=step_cap,\n    )\n    return CIDMFoundationCore(cfg)\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n\n    # 1D smoke\n    m1 = build_cidm_foundation_core(\n        spatial_dim=1,\n        channels=1,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x1 = torch.randn(2, 4, 1, 64)\n    y1 = m1(x1)\n    assert y1["pred"].shape == (2, 6, 1, 64)\n\n    # 2D smoke\n    m2 = build_cidm_foundation_core(\n        spatial_dim=2,\n        channels=2,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x2 = torch.randn(2, 4, 2, 24, 24)\n    y2 = m2(x2)\n    assert y2["pred"].shape == (2, 6, 2, 24, 24)\n\n    print("CIDM Foundation Core v1.0 smoke PASS")\n    print("1D params:", count_parameters(m1))\n    print("2D params:", count_parameters(m2))\n'
CIDM_FREEZE_MANIFEST = '{\n  "name": "CIDM Foundation Core",\n  "version": "v1.0 Final",\n  "freeze_basis": "A1-R1-R3 Core Component Causal Attribution & Freeze",\n  "freeze_gate": "PASS_TO_CORE_FINAL_FREEZE",\n  "default_core": {\n    "encoders": [\n      "SpaceEncoder",\n      "TimeHistoryEncoder"\n    ],\n    "fusion": "InformationStateFusion",\n    "dynamics": "BoundedLocalResidualDynamics",\n    "step": "bounded positive residual step",\n    "decoder": "FieldDecoder",\n    "rollout": "pure latent rollout",\n    "default_training": "prediction / multi-step rollout loss"\n  },\n  "evidence_summary": {\n    "NoSpaceInfo_mean_rmse_effect_pct": 4.895219760746574,\n    "NoTimeInfo_mean_rmse_effect_pct": 5.171493884445902,\n    "NoPhaseInfo_mean_rmse_effect_pct": -2.79857874424354,\n    "NoSkew_mean_rmse_effect_pct": -0.521169227349116,\n    "NoFlux_mean_rmse_effect_pct": -0.9437659673091198,\n    "NoLocal_mean_rmse_effect_pct": 2.397827861124943,\n    "BlackboxCore_mean_rmse_effect_pct": 1.7607338250780888,\n    "DirectPredictor_mean_rmse_effect_pct": 14.230675545900779,\n    "FreeChannelMix_vs_Skew_rmse_change_pct": 3.476865940431788,\n    "FreeFlux_vs_AntisymmetricFlux_rmse_change_pct": -2.5548147465622244,\n    "Sync05_vs_PredOnly_rmse_change_pct": 10.445702293374174,\n    "Sync05_sync_gain_change": 0.5611278432875664\n  },\n  "important_cautions": [\n    "DirectPredictor in A1-R1-R3 was materially smaller than Full, so its 14.23% RMSE gap is supportive rather than a parameter-matched causal estimate.",\n    "Only two seeds were used; several component effects had high seed variance.",\n    "Space/Time encoders show cross-task value but Diffusion-Reaction did not consistently require them.",\n    "The numerical cap values are architectural hyperparameters, not physical constants."\n  ],\n  "optional_components": {\n    "PhaseSpectralEncoder": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "NoPhaseInfo improved mean RMSE by 2.80%; benefits were inconsistent across tasks/seeds."\n    },\n    "SkewSymmetricChannelRotation": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "Removing skew slightly improved mean RMSE; however skew-constrained mixing was better than free channel mixing when a mixing branch was present."\n    },\n    "AntisymmetricFlux": {\n      "status": "SPECIALIZED CONSERVATIVE-EXCHANGE OPTIONAL",\n      "reason": "NoFlux slightly improved mean RMSE and FreeFlux outperformed antisymmetric flux on average."\n    },\n    "HiddenReservoir": {\n      "status": "OPTIONAL",\n      "reason": "Use for unresolved memory/internal closure; not part of the R3 universal core."\n    },\n    "NativeCausalPort": {\n      "status": "OPTIONAL",\n      "reason": "Use only when known causal forcing is available."\n    },\n    "Transport": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for displacement/coherent-structure dominated systems."\n    },\n    "CrossScaleTransfer": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for tasks with explicit cross-scale transfer."\n    },\n    "StateSyncRegularizer": {\n      "status": "TRAINING OPTIONAL",\n      "reason": "Improved sync metric but increased mean RMSE by 10.45% in R3."\n    },\n    "CayleySkewUpdate": {\n      "status": "NUMERICAL OPTIONAL",\n      "reason": "Strictly orthogonal discretization if a skew branch is deliberately enabled."\n    },\n    "BER_SOC": {\n      "status": "SPECIALIZED EXTENSION FAMILY",\n      "reason": "Keep outside Foundation Core; appropriate for open/physical closure studies."\n    }\n  }\n}'
CIDM_COMPLETE_ARCHIVE_SOURCE = '"""\nCIDM architecture archive\n=========================\n\nReference PyTorch implementation for the finalized model family:\n\n1. CIDM / Closed Information Dynamics Model\n   - General-purpose closed information dynamics backbone.\n   - Corresponds conceptually to the previous experimental `sticm_v3_full`.\n\n2. CIDM-BER / CIDM with Boundary Energy Reservoir\n   - Physics-closure enhanced variant.\n   - Adds a latent Spectral Operator Core and a Boundary Energy Reservoir (BER)\n     residual closure layer.\n   - Corresponds conceptually to the previous experimental `sticm_soc_ber`.\n\nThis file is intended for architecture archiving and future experiment reuse. It\ncontains only model definitions and lightweight diagnostics; training/evaluation\nloops are intentionally kept outside this file.\n\nExpected input format\n---------------------\n`history`: Tensor[B, L, C, H, W]\n`port`: optional Tensor[B, C_port, H, W] or Tensor[B, horizon, C_port, H, W]\n`target`: optional Tensor[B, horizon, C, H, W] for diagnostics/loss construction\n\nOutput dictionary\n-----------------\n`pred`: Tensor[B, horizon, C, H, W]\n`latent`: final latent state\n`latent_seq`: Tensor[B, horizon, D, H, W]\n`diagnostics`: scalar tensors for energy drift, SOC strength, BER strength, etc.\n\nAuthoring note\n--------------\nThis is a cleaned reference architecture. Exact Colab benchmark implementations\nmay contain additional data-generation, plotting, logging, and ablation utilities.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass CIDMConfig:\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n    port_channels: int = 1\n    use_phase_encoder: bool = True\n    use_time_encoder: bool = True\n    use_hidden_reservoir: bool = True\n    use_flux: bool = True\n    use_reverse_diagnostics: bool = True\n    # CIDM-BER options\n    soc_modes1: int = 8\n    soc_modes2: int = 8\n    ber_add_scale: float = 0.05\n    ber_enabled: bool = True\n    soc_enabled: bool = True\n\n\ndef count_parameters(model: nn.Module) -> int:\n    """Return the number of trainable parameters."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef conv_block(in_ch: int, out_ch: int, hidden: int, groups: int = 1) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv2d(in_ch, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, out_ch, 1),\n    )\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field into latent information state."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode a finite observation history as temporal context."""\n\n    def __init__(self, channels: int, hist_len: int, latent_dim: int):\n        super().__init__()\n        self.hist_len = hist_len\n        self.net = nn.Sequential(\n            nn.Conv2d(channels * hist_len, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: B, L, C, H, W\n        b, l, c, h, w = history.shape\n        if l != self.hist_len:\n            raise ValueError(f"Expected hist_len={self.hist_len}, got {l}.")\n        return self.net(history.reshape(b, l * c, h, w))\n\n\nclass PhaseSpectralEncoder(nn.Module):\n    """FFT-based phase/amplitude encoder projected back to the spatial latent grid."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.freq_net = nn.Sequential(\n            nn.Conv2d(channels * 3, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.spatial_net = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(x, dim=(-2, -1), norm="ortho")\n        amp = torch.log1p(torch.abs(z))\n        feats = torch.cat([z.real, z.imag, amp], dim=1)\n        f = self.freq_net(feats)\n        # Return to the spatial support while retaining frequency-aware features.\n        f_complex = torch.complex(f, torch.zeros_like(f))\n        spatial = torch.fft.ifft2(f_complex, dim=(-2, -1), norm="ortho").real\n        return self.spatial_net(spatial)\n\n\nclass FieldDecoder(nn.Module):\n    """Decode latent information state to physical/observed field."""\n\n    def __init__(self, latent_dim: int, channels: int, hidden: int = 64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, channels, 1),\n        )\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass SpectralFieldDecoder(nn.Module):\n    """Small spectral residual decoder used by CIDM for phase-aware readout."""\n\n    def __init__(self, latent_dim: int, channels: int):\n        super().__init__()\n        self.mix = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, channels, 1),\n        )\n        self.scale = nn.Parameter(torch.tensor(-2.2))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(psi, dim=(-2, -1), norm="ortho")\n        # Simple low-pass emphasis for phase-stable readout.\n        H, W = psi.shape[-2:]\n        ky = torch.fft.fftfreq(H, device=psi.device).view(1, 1, H, 1)\n        kx = torch.fft.fftfreq(W, device=psi.device).view(1, 1, 1, W)\n        filt = torch.exp(-6.0 * (kx ** 2 + ky ** 2))\n        low = torch.fft.ifft2(z * filt, dim=(-2, -1), norm="ortho").real\n        return torch.sigmoid(self.scale) * self.mix(low)\n\n\nclass AntisymmetricFlux(nn.Module):\n    """Conservative local exchange field.\n\n    For each neighbor pair, the edge update is constructed as f(a,b)-f(b,a),\n    then added to one side and subtracted from the other. This encodes the idea\n    that local exchanges can redistribute information without creating it.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.edge_net = nn.Sequential(\n            nn.Conv2d(2 * latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n\n    def edge(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n        return self.edge_net(torch.cat([a, b], dim=1)) - self.edge_net(torch.cat([b, a], dim=1))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        out = torch.zeros_like(psi)\n        # x direction\n        right = torch.roll(psi, shifts=-1, dims=-1)\n        fx = self.edge(psi, right)\n        out = out + fx - torch.roll(fx, shifts=1, dims=-1)\n        # y direction; safe for H=1 because roll is identity and antisymmetry cancels.\n        down = torch.roll(psi, shifts=-1, dims=-2)\n        fy = self.edge(psi, down)\n        out = out + fy - torch.roll(fy, shifts=1, dims=-2)\n        return out\n\n\nclass SkewChannelRotation(nn.Module):\n    """Learned skew-symmetric channel mixing for approximately conservative flow."""\n\n    def __init__(self, latent_dim: int):\n        super().__init__()\n        self.raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        A = self.raw - self.raw.t()\n        return torch.einsum("ij,bjhw->bihw", A, psi)\n\n\nclass HiddenReservoirTransfer(nn.Module):\n    """Internal hidden reservoir transfer for unobserved latent exchange."""\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = conv_block(latent_dim, latent_dim, hidden)\n        self.scale = nn.Parameter(torch.tensor(-2.0))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.scale) * self.net(psi)\n\n\nclass ClosedInformationDynamicsCore(nn.Module):\n    """CIDM closed latent vector field."""\n\n    def __init__(self, latent_dim: int, hidden: int, use_flux: bool = True, use_hidden_reservoir: bool = True):\n        super().__init__()\n        self.use_flux = use_flux\n        self.use_hidden_reservoir = use_hidden_reservoir\n        self.skew = SkewChannelRotation(latent_dim)\n        self.flux = AntisymmetricFlux(latent_dim, hidden)\n        self.hidden = HiddenReservoirTransfer(latent_dim, hidden)\n        self.local = conv_block(latent_dim, latent_dim, hidden)\n        self.step_scale = nn.Parameter(torch.tensor(0.18))\n\n    def vector_field(self, psi: torch.Tensor) -> torch.Tensor:\n        dpsi = self.skew(psi)\n        if self.use_flux:\n            dpsi = dpsi + self.flux(psi)\n        if self.use_hidden_reservoir:\n            dpsi = dpsi + 0.25 * self.hidden(psi)\n        # A small local residual stabilizes non-Hamiltonian observed fields.\n        dpsi = dpsi + 0.10 * self.local(psi)\n        return dpsi\n\n    def forward(self, psi: torch.Tensor, dt_scale: float = 1.0) -> torch.Tensor:\n        alpha = torch.tanh(self.step_scale) * dt_scale\n        return psi + alpha * self.vector_field(psi)\n\n\nclass LatentSpectralOperatorCore(nn.Module):\n    """Latent-domain spectral operator for CIDM-BER.\n\n    This is a residual spectral vector field inside the latent dynamics, not an\n    external FNO baseline. It learns complex low-frequency mode mixing and maps\n    it back to the latent grid as a global correction to local closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int = 64, modes1: int = 8, modes2: int = 8):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.modes1 = modes1\n        self.modes2 = modes2\n        scale = 1.0 / max(1, latent_dim)\n        self.weight_pos = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.weight_neg = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.pre = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.post = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-1.8))\n\n    @staticmethod\n    def compl_mul2d(x: torch.Tensor, wri: torch.Tensor) -> torch.Tensor:\n        w = torch.view_as_complex(wri.contiguous())\n        return torch.einsum("bihw,iohw->bohw", x, w)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        B, D, H, W = psi.shape\n        x = self.pre(psi)\n        x_ft = torch.fft.rfft2(x, dim=(-2, -1), norm="ortho")\n        out_ft = torch.zeros(B, D, H, W // 2 + 1, dtype=torch.cfloat, device=psi.device)\n        m1 = min(self.modes1, H)\n        m2 = min(self.modes2, W // 2 + 1)\n        out_ft[:, :, :m1, :m2] = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weight_pos[:, :, :m1, :m2])\n        if H > 1:\n            out_ft[:, :, -m1:, :m2] = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weight_neg[:, :, :m1, :m2])\n        y = torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1), norm="ortho")\n        return torch.sigmoid(self.logit_scale) * self.post(y)\n\n\nclass BoundaryEnergyReservoir(nn.Module):\n    """Boundary Energy Reservoir (BER) residual closure layer.\n\n    BER is a virtual external energy reservoir, not a full external-state model.\n    It learns a minimal residual exchange field that accounts for open-system\n    effects not captured by the internal closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, port_channels: int, hidden: int):\n        super().__init__()\n        self.port_channels = max(1, port_channels)\n        in_ch = latent_dim * 2 + self.port_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 3, padding=1),\n        )\n        mask_hidden = max(8, hidden // 2)\n        self.mask = nn.Sequential(\n            nn.Conv2d(in_ch, mask_hidden, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(mask_hidden, 1, 1),\n            nn.Sigmoid(),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-3.0))\n\n    def _port_to_grid(self, psi: torch.Tensor, port: Optional[torch.Tensor]) -> torch.Tensor:\n        B, _, H, W = psi.shape\n        if port is None or port.numel() == 0:\n            return torch.zeros(B, self.port_channels, H, W, device=psi.device, dtype=psi.dtype)\n        if port.dim() != 4:\n            raise ValueError("port must be Tensor[B, C_port, H, W] for one step.")\n        p = port\n        if p.shape[-2:] != (H, W):\n            p = F.interpolate(p, size=(H, W), mode="bilinear", align_corners=False)\n        if p.shape[1] < self.port_channels:\n            pad = torch.zeros(B, self.port_channels - p.shape[1], H, W, device=psi.device, dtype=psi.dtype)\n            p = torch.cat([p, pad], dim=1)\n        elif p.shape[1] > self.port_channels:\n            p = p[:, : self.port_channels]\n        return p\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n        internal_delta: torch.Tensor,\n        port: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        p = self._port_to_grid(psi, port)\n        x = torch.cat([psi, internal_delta, p], dim=1)\n        raw = torch.tanh(self.net(x))\n        mask = self.mask(x)\n        scale = torch.sigmoid(self.logit_scale)\n        ber_delta = scale * mask * raw\n        info = {\n            "ber_strength": ber_delta.abs().mean(),\n            "ber_mask_mean": mask.mean(),\n            "ber_scale": scale,\n        }\n        return ber_delta, info\n\n\nclass CIDM(nn.Module):\n    """Closed Information Dynamics Model.\n\n    General-purpose主体架构. This version corresponds to the stabilized closed\n    dynamics backbone: spatial information encoding, temporal history encoding,\n    phase-spectral feature encoding, conservative local exchange, internal hidden\n    reservoir transfer, and autoregressive latent rollout.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__()\n        self.config = config\n        D = config.latent_dim\n        H = config.hidden\n        self.space_encoder = SpaceEncoder(config.channels, D)\n        self.time_encoder = TimeHistoryEncoder(config.channels, config.hist_len, D)\n        self.phase_encoder = PhaseSpectralEncoder(config.channels, D)\n        fuse_inputs = D + (D if config.use_time_encoder else 0) + (D if config.use_phase_encoder else 0)\n        self.fuse = nn.Sequential(nn.Conv2d(fuse_inputs, D, 1), nn.SiLU(), nn.Conv2d(D, D, 1))\n        self.core = ClosedInformationDynamicsCore(D, H, use_flux=config.use_flux, use_hidden_reservoir=config.use_hidden_reservoir)\n        self.decoder = FieldDecoder(D, config.channels, H)\n        self.spectral_decoder = SpectralFieldDecoder(D, config.channels)\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        feats = [self.space_encoder(last)]\n        if self.config.use_time_encoder:\n            feats.append(self.time_encoder(history))\n        if self.config.use_phase_encoder:\n            feats.append(self.phase_encoder(last))\n        return self.fuse(torch.cat(feats, dim=1))\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi) + self.spectral_decoder(psi)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_next = self.core(psi)\n        return psi_next, {"soc_strength": torch.zeros((), device=psi.device), "ber_strength": torch.zeros((), device=psi.device)}\n\n    def _select_port_step(self, port: Optional[torch.Tensor], t: int) -> Optional[torch.Tensor]:\n        if port is None:\n            return None\n        if port.dim() == 5:\n            return port[:, min(t, port.shape[1] - 1)]\n        return port\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        port: Optional[torch.Tensor] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        horizon = int(horizon or self.config.horizon)\n        psi0 = self.encode(history)\n        psi = psi0\n        preds = []\n        latent_seq = []\n        soc_strengths, ber_strengths = [], []\n        energy0 = (psi0 ** 2).mean(dim=(1, 2, 3))\n        energies = []\n        for t in range(horizon):\n            psi, info = self.step(psi, t, self._select_port_step(port, t))\n            y = self.decode(psi)\n            preds.append(y)\n            latent_seq.append(psi)\n            energies.append((psi ** 2).mean(dim=(1, 2, 3)))\n            soc_strengths.append(info.get("soc_strength", torch.zeros((), device=psi.device)))\n            ber_strengths.append(info.get("ber_strength", torch.zeros((), device=psi.device)))\n        pred = torch.stack(preds, dim=1)\n        latent = torch.stack(latent_seq, dim=1)\n        energy = torch.stack(energies, dim=1)\n        diagnostics = {\n            "energy_drift": (energy[:, -1] - energy0).abs().mean(),\n            "soc_strength": torch.stack([s if s.dim() else s.expand(1) for s in soc_strengths]).mean(),\n            "ber_strength": torch.stack([b if b.dim() else b.expand(1) for b in ber_strengths]).mean(),\n        }\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n        return {"pred": pred, "latent": psi, "latent_seq": latent, "diagnostics": diagnostics}\n\n\nclass CIDM_BER(CIDM):\n    """CIDM with Spectral Operator Core and Boundary Energy Reservoir.\n\n    Physics-enhanced variant for systems with salient wave/flow propagation,\n    quasi-closed dynamics, boundary exchange, or energy closure requirements.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__(config)\n        self.soc = LatentSpectralOperatorCore(\n            config.latent_dim,\n            hidden=config.hidden,\n            modes1=config.soc_modes1,\n            modes2=config.soc_modes2,\n        )\n        self.ber = BoundaryEnergyReservoir(config.latent_dim, config.port_channels, config.hidden)\n        self.ber_add_scale = float(config.ber_add_scale)\n        self.soc_enabled = bool(config.soc_enabled)\n        self.ber_enabled = bool(config.ber_enabled)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_int = self.core(psi)\n        soc_delta = self.soc(psi) if self.soc_enabled else torch.zeros_like(psi)\n        psi_int = psi_int + soc_delta\n        internal_delta = psi_int - psi\n        if self.ber_enabled:\n            ber_delta, ber_info = self.ber(psi, internal_delta.detach(), port)\n            psi_next = psi_int + self.ber_add_scale * ber_delta\n        else:\n            ber_delta = torch.zeros_like(psi)\n            ber_info = {"ber_strength": torch.zeros((), device=psi.device), "ber_mask_mean": torch.zeros((), device=psi.device)}\n            psi_next = psi_int\n        info = {\n            "soc_strength": soc_delta.abs().mean(),\n            "ber_strength": ber_info["ber_strength"],\n            "ber_mask_mean": ber_info.get("ber_mask_mean", torch.zeros((), device=psi.device)),\n            "ber_budget_proxy": (ber_delta.mean(dim=(1, 2, 3)) - internal_delta.mean(dim=(1, 2, 3))).abs().mean(),\n        }\n        return psi_next, info\n\n\ndef build_cidm(config: Optional[CIDMConfig] = None) -> CIDM:\n    return CIDM(config or CIDMConfig())\n\n\ndef build_cidm_ber(config: Optional[CIDMConfig] = None) -> CIDM_BER:\n    cfg = config or CIDMConfig()\n    return CIDM_BER(cfg)\n\n\n# Backward-compatible aliases for previous internal experiment names.\nClosedInformationDynamicsModel = CIDM\nClosedInformationDynamicsBERModel = CIDM_BER\nSTICM_V3_FULL_ALIAS = CIDM\nSTICM_SOC_BER_ALIAS = CIDM_BER\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    cfg = CIDMConfig(channels=1, hist_len=4, horizon=1, latent_dim=4, hidden=8, port_channels=1)\n    x = torch.randn(1, cfg.hist_len, cfg.channels, 4, 4)\n    port = torch.randn(1, cfg.port_channels, 4, 4)\n    for cls in [CIDM, CIDM_BER]:\n        model = cls(cfg)\n        out = model(x, port=port)\n        print(cls.__name__, out["pred"].shape, "params=", count_parameters(model), out["diagnostics"])\n'

EXPECTED_CORE_SHA256 = "e5e26b35765301e9ecd25aec5a6c507395ff370dca11c820f540ff0f1e79f510"
EXPECTED_MANIFEST_SHA256 = "9508dd000005a154d3d3fa4437602d4214a6266d630bf13af855e5517649ecbd"
EXPECTED_COMPLETE_ARCHIVE_SHA256 = "f9b1cf4ff7630f5540bb3adde20e4c4b81fff3218d9325ea3105bf28b1d97665"

core_file=FROZEN/"cidm_foundation_core_v1.py"
manifest_file=FROZEN/"CIDM_Foundation_Core_v1_0_FREEZE_MANIFEST.json"
complete_file=FROZEN/"cidm_complete_archived_topology.py"

core_file.write_text(CIDM_CORE_SOURCE,encoding="utf-8")
manifest_file.write_text(CIDM_FREEZE_MANIFEST,encoding="utf-8")
complete_file.write_text(CIDM_COMPLETE_ARCHIVE_SOURCE,encoding="utf-8")

core_sha=sha256_file(core_file)
manifest_sha=sha256_file(manifest_file)
complete_sha=sha256_file(complete_file)

print("CIDM Core SHA256:",core_sha)
print("Core Manifest SHA256:",manifest_sha)
print("CIDM-Complete archive SHA256:",complete_sha)

assert core_sha==EXPECTED_CORE_SHA256
assert manifest_sha==EXPECTED_MANIFEST_SHA256
assert complete_sha==EXPECTED_COMPLETE_ARCHIVE_SHA256

import importlib.util

spec=importlib.util.spec_from_file_location("cidm_foundation_core_v1",core_file)
cidm_mod=importlib.util.module_from_spec(spec)
sys.modules["cidm_foundation_core_v1"]=cidm_mod
spec.loader.exec_module(cidm_mod)

CIDMFoundationCore=cidm_mod.CIDMFoundationCore
CIDMFoundationConfig=cidm_mod.CIDMFoundationConfig
build_cidm_foundation_core=cidm_mod.build_cidm_foundation_core

spec2=importlib.util.spec_from_file_location("cidm_complete_archive",complete_file)
cidm_complete_mod=importlib.util.module_from_spec(spec2)
sys.modules["cidm_complete_archive"]=cidm_complete_mod
spec2.loader.exec_module(cidm_complete_mod)

CIDMCompleteConfig=cidm_complete_mod.CIDMConfig
CIDMCompleteArchived=cidm_complete_mod.CIDM_BER

# Core smoke
m=build_cidm_foundation_core(
    spatial_dim=2,channels=1,hist_len=4,horizon=6,
    latent_dim=16,hidden=32
).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 2D smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=2,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 1D-via-H=1 smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=1,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,1,64,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,1,64)
del m,x,y
torch.cuda.empty_cache()

COMPLETE_CONTRACT={
    "benchmark_name":"CIDM-Complete",
    "executable_mapping":"cidm_architectures.py::CIDM_BER",
    "archive_sha256":complete_sha,
    "components":[
        "SpaceEncoder","TimeHistoryEncoder","PhaseSpectralEncoder",
        "SkewChannelRotation","AntisymmetricFlux","HiddenReservoirTransfer",
        "LocalResidual","LatentSpectralOperatorCore","BoundaryEnergyReservoir",
        "FieldDecoder","SpectralFieldDecoder","PureLatentRollout"
    ],
    "known_forcing_port":"disabled / None because PDEBench headline tasks provide no common forcing field",
    "static_fields":"not supplied; absent from archived executable implementation",
    "1d_protocol":"lift to H=1, run exact archived Conv2D implementation, squeeze H=1",
    "future_target_at_inference":"forbidden",
    "note":"Figure-level State-Observation Closure is not invented here; archived executable SOC is LatentSpectralOperatorCore."
}
save_json(COMPLETE_CONTRACT,OUT/"CIDM_Complete_Benchmark_Contract.json")

REPORT["cidm_freeze"]={
    "core_sha256":core_sha,
    "manifest_sha256":manifest_sha,
    "exact":True,
}
REPORT["cidm_complete_freeze"]={
    "archive_sha256":complete_sha,
    "exact":True,
    "contract":COMPLETE_CONTRACT,
}

print("FROZEN CIDM CORE: PASS")
print("FROZEN CIDM-COMPLETE ARCHIVE: PASS")


## 2. Official source provenance

In [ ]:

# ============================================================
# 2. Exact source checkout + official imports
# ============================================================
LOCKED={
    "pdebench":{
        "url":"https://github.com/pdebench/PDEBench.git",
        "sha":"4ff3e3a4aa1561721b5571fa3a048a0a463e0568",
    },
    "neuraloperator":{
        "url":"https://github.com/neuraloperator/neuraloperator.git",
        "sha":"00b7d86f8d74ff0af55da53eb585fe26df9c71f0",
    },
    "transolver":{
        "url":"https://github.com/thuml/Transolver.git",
        "sha":"75e0f67643806a81cd1d3f6adc88dd8c02416fe7",
    },
}
checkout=[]
for rid,m in LOCKED.items():
    dst=SRC/rid
    if dst.exists():
        rc,head=run(["git","rev-parse","HEAD"],cwd=dst,timeout=60)
        if rc==0 and head.strip()==m["sha"]:
            print("[reuse source]",rid,head.strip())
            checkout.append({"id":rid,"head":head.strip(),"exact":True})
            continue
        shutil.rmtree(dst)

    rc,o=run(["git","clone","--filter=blob:none","--no-checkout",m["url"],str(dst)])
    if rc: raise RuntimeError(f"{rid} clone failed\n{o[-3000:]}")
    rc,o=run(["git","fetch","origin",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} fetch failed\n{o[-3000:]}")
    rc,o=run(["git","checkout","--detach",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} checkout failed\n{o[-3000:]}")
    _,head=run(["git","rev-parse","HEAD"],cwd=dst)
    head=head.strip()
    checkout.append({"id":rid,"head":head,"exact":head==m["sha"]})
    print(rid,head,"PASS" if head==m["sha"] else "FAIL")

assert all(x["exact"] for x in checkout)
pd.DataFrame(checkout).to_csv(OUT/"source_checkout.csv",index=False)

# Dependencies needed by the exact official cores.
rc,o=run([
    sys.executable,"-m","pip","install","-q",
    "tensorly","tensorly-torch","einops","timm","scipy"
],timeout=1200)
if rc:
    raise RuntimeError(o[-5000:])

sys.path.insert(0,str(SRC/"pdebench"))
sys.path.insert(0,str(SRC/"neuraloperator"))
TRANS_BENCH=SRC/"transolver"/"PDE-Solving-StandardBenchmark"
sys.path.insert(0,str(TRANS_BENCH))

from pdebench.models.fno.fno import FNO1d as PDE_FNO1d, FNO2d as PDE_FNO2d
from pdebench.models.unet.unet import UNet1d as PDE_UNet1d, UNet2d as PDE_UNet2d
from neuralop.models import FNO as NeuralOpFNO
from model.Transolver_Irregular_Mesh import Model as TransolverIrregular
from model.Transolver_Structured_Mesh_2D import Model as Transolver2D

print("OFFICIAL IMPORTS: PASS")
REPORT["source_checkout"]=checkout


## 3. Official wrappers

In [ ]:

# ============================================================
# 4. Official wrappers
# ============================================================
def normalized_grid(spatial,device):
    if len(spatial)==1:
        return torch.linspace(0,1,spatial[0],device=device)[:,None]
    x=torch.linspace(0,1,spatial[0],device=device)
    y=torch.linspace(0,1,spatial[1],device=device)
    X,Y=torch.meshgrid(x,y,indexing="ij")
    return torch.stack([X,Y],-1)

class CIDMWrapper(nn.Module):
    def __init__(self,dim,c,L,K,D,H):
        super().__init__()
        self.model=build_cidm_foundation_core(
            spatial_dim=dim,channels=c,hist_len=L,horizon=K,
            latent_dim=D,hidden=H
        )
    def rollout(self,h,K):
        return self.model(h,horizon=K)["pred"]


class CIDMCompleteWrapper(nn.Module):
    """
    Exact archived CIDM_BER topology used as CIDM-Complete.
    For 1D tasks we lift X -> [1,X] and use the same archived Conv2D model.
    No external forcing/port is supplied in the PDEBench headline panel.
    """
    def __init__(self,dim,c,L,K,D,H,modes1,modes2):
        super().__init__()
        self.dim=dim
        self.config=CIDMCompleteConfig(
            channels=c,
            hist_len=L,
            horizon=K,
            latent_dim=D,
            hidden=H,
            port_channels=1,
            use_phase_encoder=True,
            use_time_encoder=True,
            use_hidden_reservoir=True,
            use_flux=True,
            use_reverse_diagnostics=True,
            soc_modes1=modes1,
            soc_modes2=modes2,
            ber_add_scale=0.05,
            ber_enabled=True,
            soc_enabled=True,
        )
        self.model=CIDMCompleteArchived(self.config)

    def _lift(self,h):
        return h.unsqueeze(-2) if self.dim==1 else h

    def rollout(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        p=out["pred"]
        return p.squeeze(-2) if self.dim==1 else p

    @torch.no_grad()
    def architecture_diagnostics(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        d=out["diagnostics"]
        return {
            "latent_energy_drift":float(d["energy_drift"].detach()),
            "soc_strength":float(d["soc_strength"].detach()),
            "ber_strength":float(d["ber_strength"].detach()),
        }


class PDEBenchFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        if dim==1:
            self.model=PDE_FNO1d(c,modes=modes,width=width,initial_step=L)
        else:
            self.model=PDE_FNO2d(c,modes1=modes,modes2=modes,width=width,initial_step=L)
    def one(self,h):
        B,L,C,*S=h.shape
        if self.dim==1:
            inp=h.permute(0,3,1,2).reshape(B,S[0],L*C)
        else:
            inp=h.permute(0,3,4,1,2).reshape(B,S[0],S[1],L*C)
        grid=normalized_grid(S,h.device)[None].expand(B,*([-1]*self.dim),-1)
        y=self.model(inp,grid).squeeze(-2)
        if self.dim==1: return y.permute(0,2,1).contiguous()
        return y.permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class PDEBenchUNetWrapper(nn.Module):
    def __init__(self,dim,c,L,features):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=(PDE_UNet1d if dim==1 else PDE_UNet2d)(
            in_channels=c*L,out_channels=c,init_features=features
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class NeuralOpFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=NeuralOpFNO(
            n_modes=tuple([modes]*dim),
            in_channels=c*L,out_channels=c,
            hidden_channels=width,n_layers=4,
            positional_embedding="grid",
            fno_block_precision="full",
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class TransolverWrapper(nn.Module):
    def __init__(self,dim,c,L,S,n_hidden,n_layers,n_head,slice_num):
        super().__init__(); self.dim=dim; self.c=c; self.L=L; self.S=S
        kw=dict(
            space_dim=dim,n_layers=n_layers,n_hidden=n_hidden,dropout=0.0,
            n_head=n_head,Time_Input=False,act="gelu",mlp_ratio=1,
            fun_dim=c*L,out_dim=c,slice_num=slice_num,unified_pos=False
        )
        if dim==1:
            self.model=TransolverIrregular(**kw)
        else:
            self.model=Transolver2D(**kw,H=S[0],W=S[1])
    def one(self,h):
        B,L,C,*S=h.shape
        g=normalized_grid(S,h.device).reshape(-1,self.dim)[None].expand(B,-1,-1)
        if self.dim==1:
            fx=h.permute(0,3,1,2).reshape(B,-1,L*C)
        else:
            fx=h.permute(0,3,4,1,2).reshape(B,-1,L*C)
        y=self.model(g,fx)
        if self.dim==1:
            return y.reshape(B,S[0],C).permute(0,2,1).contiguous()
        return y.reshape(B,S[0],S[1],C).permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)


In [ ]:

# ============================================================
# 3.1 Frozen constructors
# ============================================================
def build_model(task,name):
    meta=TASKS[task]; c=meta["channels"]; dim=meta["dim"]; S=tuple(meta["spatial"])
    cfg=CONFIG[task][name]
    if name=="CIDM-Foundation-v1":
        return CIDMWrapper(dim,c,HISTORY,TRAIN_HORIZON,cfg["latent_dim"],cfg["hidden"])
    if name=="CIDM-Complete":
        return CIDMCompleteWrapper(
            dim,c,HISTORY,TRAIN_HORIZON,cfg["latent_dim"],cfg["hidden"],
            cfg["soc_modes1"],cfg["soc_modes2"]
        )
    if name=="PDEBench-FNO":
        return PDEBenchFNOWrapper(dim,c,HISTORY,cfg["width"],cfg["modes"])
    if name=="NeuralOperator-FNO":
        return NeuralOpFNOWrapper(dim,c,HISTORY,cfg["width"],cfg["modes"])
    raise KeyError(name)

param_rows=[]
for t in TASKS:
    for m in MODELS:
        q=build_model(t,m)
        param_rows.append({"task":t,"model":m,"params":count_params(q)})
        del q
param_df=pd.DataFrame(param_rows)
param_df.to_csv(OUT/"A4_R1_parameter_audit.csv",index=False)
display(param_df)
assert ((param_df.params>=475000)&(param_df.params<=525000)).all()


## 4. HF Dataset persistence + restore exact A2 training caches

In [ ]:

# ============================================================
# 4. HF DATASET runtime
# ============================================================
HF_REPO=os.getenv("CIDM_HF_DATASET_REPO","wuff-mann/CIDM-v1").strip()
A2_PREFIX="A2_BroadDynamics_500K_3Seed"
HF_PREFIX="A4_R1_LongRollout_Generalization_3Seed"
HF_CHECKPOINT_EVERY=int(os.getenv("CIDM_HF_CHECKPOINT_EVERY","500"))
HF_MAX_RETRIES=4

try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN=os.getenv("HF_TOKEN")
if not HF_TOKEN: raise RuntimeError("HF_TOKEN required.")

subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub"],check=True)
from huggingface_hub import HfApi,hf_hub_download,CommitOperationAdd
api=HfApi(token=HF_TOKEN)

def retry(fn,*args,**kwargs):
    last=None
    for a in range(1,HF_MAX_RETRIES+1):
        try: return fn(*args,**kwargs)
        except Exception as e:
            last=e
            if a==HF_MAX_RETRIES: break
            time.sleep(min(30,2**a))
    raise last

retry(api.dataset_info,repo_id=HF_REPO,revision="main")
REMOTE=retry(api.list_repo_files,repo_id=HF_REPO,repo_type="dataset",revision="main")

def dl(remote):
    return Path(retry(
        hf_hub_download,repo_id=HF_REPO,filename=remote,repo_type="dataset",
        revision="main",token=HF_TOKEN,local_dir="/content/_cidm_a4_restore"
    ))

# A2 train caches
CACHE_FILES={}
for task in TASKS:
    cand=[f for f in REMOTE if f.startswith(A2_PREFIX+"/cache/") and Path(f).name.startswith(task+"_") and f.endswith(".pt")]
    if len(cand)!=1: raise RuntimeError((task,cand))
    src=dl(cand[0]); dst=CACHE/Path(cand[0]).name
    if not dst.exists(): shutil.copy2(src,dst)
    CACHE_FILES[task]=dst

def scan_progress():
    rows=[]
    for p in OUT.glob("A4R1__*_result.json"):
        try: rows.append(json.loads(p.read_text(encoding="utf-8")))
        except: pass
    return {"expected":48,"completed":sum(r.get("status")=="COMPLETED" for r in rows),
            "result_files":len(rows),"checkpoints":len(list(CKPT.glob("A4R1__*_latest.pt")))}

def manifest(run_id=None,step=None,event=None):
    x=scan_progress()
    x.update({"repo_id":HF_REPO,"prefix":HF_PREFIX,"updated_at":datetime.now(timezone.utc).isoformat(),
              "current_run":run_id,"current_step":step,"event":event})
    p=OUT/"A4_R1_progress_manifest.json"
    p.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding="utf-8")
    return p,x

def repo_path(p):
    p=Path(p).resolve()
    if LONGCACHE.resolve() in p.parents:
        return f"{HF_PREFIX}/long_eval_cache/{p.relative_to(LONGCACHE.resolve()).as_posix()}"
    return f"{HF_PREFIX}/{p.relative_to(OUT.resolve()).as_posix()}"

def commit(files,msg):
    ops=[]
    for p in files:
        p=Path(p)
        if p.exists() and p.is_file():
            ops.append(CommitOperationAdd(path_in_repo=repo_path(p),path_or_fileobj=str(p)))
    if ops:
        retry(api.create_commit,repo_id=HF_REPO,repo_type="dataset",operations=ops,commit_message=msg)

def sync_ckpt(cp,run_id,step):
    man,_=manifest(run_id,step,"MID_RUN_CHECKPOINT")
    try: commit([cp,man],f"[A4-R1] checkpoint {run_id} step={step}")
    except Exception as e: print("[HF ckpt warning]",e)

def sync_done(run_id,rp,hp,cp):
    row=json.loads(Path(rp).read_text(encoding="utf-8"))
    man,state=manifest(run_id,None,f"RUN_{row.get('status')}")
    partial=[]
    for p in OUT.glob("A4R1__*_result.json"):
        try: partial.append(json.loads(p.read_text(encoding="utf-8")))
        except: pass
    pp=OUT/"A4_R1_run_results_partial.csv"
    pd.DataFrame(partial).to_csv(pp,index=False)
    files=[rp,hp,man,pp]
    if row.get("status")!="COMPLETED" and Path(cp).exists(): files.append(cp)
    try:
        commit(files,f"[A4-R1] {row.get('status')} {run_id} completed={state['completed']}/48")
        if row.get("status")=="COMPLETED":
            try: retry(api.delete_file,path_in_repo=repo_path(cp),repo_id=HF_REPO,repo_type="dataset",
                       commit_message=f"[A4-R1] cleanup {run_id}")
            except: pass
    except Exception as e: print("[HF done warning]",e)

# Restore A4 run state and long eval cache
for rf in [f for f in REMOTE if f.startswith(HF_PREFIX+"/")]:
    name=Path(rf).name
    if "/long_eval_cache/" in rf:
        dst=LONGCACHE/name
    elif name.startswith("A4R1__") and name.endswith("_result.json"):
        dst=OUT/name
    elif name.startswith("A4R1__") and name.endswith("_history.csv"):
        dst=OUT/name
    elif name.startswith("A4R1__") and name.endswith("_latest.pt"):
        dst=CKPT/name
    else:
        continue
    src=dl(rf); dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists() and dst.name.endswith("_result.json"):
        try:
            if json.loads(dst.read_text())["status"]=="COMPLETED": continue
        except: pass
    if dst.exists() and dst.name.endswith("_latest.pt"):
        l=torch.load(dst,map_location="cpu",weights_only=False)
        r=torch.load(src,map_location="cpu",weights_only=False)
        if int(l.get("step",-1))>=int(r.get("step",-1)): continue
    if not dst.exists() or dst.name.endswith(("_latest.pt","_result.json","_history.csv")):
        shutil.copy2(src,dst)

print("[A4 restore]",scan_progress())


## 5. Training cache + independent long-horizon raw-trajectory cache

In [ ]:

# ============================================================
# 5. Data
# ============================================================
CACHES={t:torch.load(p,map_location="cpu",weights_only=False) for t,p in CACHE_FILES.items()}

class Norm:
    def __init__(self,m,s):
        self.mean=m.float(); self.std=s.float().clamp_min(1e-6)
    def norm(self,x):
        shp=[1,1,-1]+[1]*(x.ndim-3)
        return (x-self.mean.view(*shp))/self.std.view(*shp)
    def denorm(self,x):
        shp=[1,1,-1]+[1]*(x.ndim-3)
        return x*self.std.view(*shp)+self.mean.view(*shp)

NORMS={t:Norm(c["norm"]["mean"],c["norm"]["std"]) for t,c in CACHES.items()}

class TrainWindows(Dataset):
    def __init__(self,c,split,n):
        self.h=c["splits"][split]["history"].float()
        self.y=c["splits"][split]["target"].float()
        self.n=n
    def __len__(self): return len(self.h)
    def __getitem__(self,i):
        h=self.h[i]; y=self.y[i]
        mh=self.n.mean.view(1,-1,*([1]*(h.ndim-2)))
        sh=self.n.std.view(1,-1,*([1]*(h.ndim-2)))
        return (h-mh)/sh,(y-mh)/sh

def loaders(task,seed):
    b=BATCH_1D if TASKS[task]["dim"]==1 else BATCH_2D
    c=CACHES[task]; n=NORMS[task]
    g=torch.Generator().manual_seed(seed)
    return (
        DataLoader(TrainWindows(c,"train",n),batch_size=b,shuffle=True,generator=g,num_workers=0,drop_last=True,pin_memory=True),
        DataLoader(TrainWindows(c,"val",n),batch_size=b,shuffle=False,num_workers=0,pin_memory=True),
    )

def resume_download(url,path,md5,chunk=8*1024*1024):
    path=Path(path)
    if path.exists() and md5_file(path)==md5: return
    part=Path(str(path)+".part")
    pos=part.stat().st_size if part.exists() else 0
    headers={"Range":f"bytes={pos}-"} if pos else {}
    with requests.get(url,stream=True,headers=headers,timeout=(30,300)) as r:
        if pos and r.status_code==200:
            part.unlink(missing_ok=True); pos=0
        r.raise_for_status()
        with open(part,"ab" if pos and r.status_code==206 else "wb") as f:
            for b in r.iter_content(chunk_size=chunk):
                if b: f.write(b)
    part.replace(path)
    if md5_file(path)!=md5: raise RuntimeError(f"MD5 mismatch {path}")

def read_raw_traj(task,path,idx):
    m=TASKS[task]
    with h5py.File(path,"r") as f:
        if "tensor" in f:
            a=np.asarray(f["tensor"][idx],dtype=np.float32) # [T,X]
            a=a[:,None,::m["stride"]]
        else:
            groups=[k for k in sorted(f.keys()) if isinstance(f[k],h5py.Group) and "data" in f[k]]
            a=np.asarray(f[groups[idx]]["data"],dtype=np.float32)
            if a.ndim==4:
                a=np.moveaxis(a,-1,1) # [T,C,H,W]
            elif a.ndim==3:
                a=a[:,None,:,:]
            else:
                raise RuntimeError((task,a.shape))
            if m["stride"]>1: a=a[...,::m["stride"],::m["stride"]]
    return torch.from_numpy(np.ascontiguousarray(a))

LONG={}
long_manifest=[]
for task,m in TASKS.items():
    cp=LONGCACHE/f"{task}_long_eval.pt"
    if cp.exists():
        d=torch.load(cp,map_location="cpu",weights_only=False)
    else:
        raw=RAW/m["filename"]
        resume_download(m["url"],raw,m["md5"])
        ids=CACHES[task]["selected_trajectory_ids"]["test"]
        trajs=[read_raw_traj(task,raw,int(i)) for i in ids]
        minT=min(x.shape[0] for x in trajs)
        max_h=minT-HISTORY
        available=[h for h in REQUESTED_HORIZONS if h<=max_h]
        if not available or max(available)<24:
            raise RuntimeError(f"{task} supports only max_h={max_h}; A4 requires >=24.")
        Hmax=max(available)
        # Use start=0 for all selected test trajectories.
        hist=torch.stack([x[:HISTORY] for x in trajs])
        target=torch.stack([x[HISTORY:HISTORY+Hmax] for x in trajs])
        d={"history":hist,"target":target,"available_horizons":available,
           "test_ids":[int(i) for i in ids],"min_T":minT}
        torch.save(d,cp)
        try: commit([cp],f"[A4-R1] long-eval cache {task}")
        except Exception as e: print("[cache upload warning]",e)
    LONG[task]=d
    long_manifest.append({"task":task,"min_T":d["min_T"],"available_horizons":d["available_horizons"],
                          "n_test":len(d["history"]),"shape":list(d["history"].shape)})

long_manifest_df=pd.DataFrame(long_manifest)
long_manifest_df.to_csv(OUT/"A4_R1_long_eval_manifest.csv",index=False)
display(long_manifest_df)

COMMON_HORIZONS=[
    h for h in REQUESTED_HORIZONS
    if all(h in LONG[t]["available_horizons"] for t in TASKS)
]
assert 24 in COMMON_HORIZONS
print("COMMON_HORIZONS:",COMMON_HORIZONS)
REPORT["common_horizons"]=COMMON_HORIZONS


## 6. Long-rollout / perturbation metrics + preflight

In [ ]:

# ============================================================
# 6. Evaluation
# ============================================================
def long_eval(model,task,history,target,horizons,noise_frac=0.0,history_mode="clean",rng_seed=0):
    model.eval()
    n=NORMS[task]
    h=history.clone().float()
    y=target.clone().float()

    # perturb history in physical space using training channel std
    if noise_frac>0:
        g=torch.Generator().manual_seed(rng_seed)
        std=n.std.view(1,1,-1,*([1]*(h.ndim-3)))
        eps=torch.randn(h.shape,generator=g,dtype=h.dtype)
        h=h+noise_frac*std*eps

    if history_mode=="drop_oldest_1":
        h[:,0]=h[:,1]
    elif history_mode=="drop_oldest_2":
        h[:,0]=h[:,2]; h[:,1]=h[:,2]
    elif history_mode=="last_frame_only":
        h=h[:,-1:].expand_as(h).clone()

    mh=n.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=n.std.view(1,1,-1,*([1]*(h.ndim-3)))
    hn=((h-mh)/sh).to(DEVICE)

    K=max(horizons)
    with torch.no_grad():
        pn=model.rollout(hn,K).cpu()
    p=pn*sh+mh

    out={}
    for K0 in horizons:
        e=p[:,:K0]-y[:,:K0]
        rmse=float(torch.sqrt((e**2).mean()))
        rel=float(
            torch.linalg.vector_norm(e.reshape(e.shape[0],-1),dim=1).div(
                torch.linalg.vector_norm(y[:,:K0].reshape(y.shape[0],-1),dim=1).clamp_min(1e-8)
            ).mean()
        )
        out[K0]={"rmse":rmse,"relative_l2":rel}
    return out

# 16 model/task preflight on A2 validation batch.
pre=[]
for task in TASKS:
    _,va=loaders(task,17)
    h,y=next(iter(va)); h=h.to(DEVICE); y=y.to(DEVICE)
    for name in MODELS:
        seed_all(17)
        m=build_model(task,name).to(DEVICE).float()
        p=m.rollout(h,TRAIN_HORIZON)
        loss=F.mse_loss(p,y)
        loss.backward()
        finite=bool(torch.isfinite(loss))
        shape=tuple(p.shape)==tuple(y.shape)
        pre.append({"task":task,"model":name,"finite":finite,"shape":shape,"pass":finite and shape})
        del m
        torch.cuda.empty_cache()
pre_df=pd.DataFrame(pre)
pre_df.to_csv(OUT/"A4_R1_preflight.csv",index=False)
assert pre_df["pass"].all()
print("A4-R1 PREFLIGHT 16/16 PASS")


## 7. 48-run formal training + long/generalization evaluation

In [ ]:

# ============================================================
# 7. Formal training
# ============================================================
def normalized_train(task):
    c=CACHES[task]; n=NORMS[task]
    h=c["splits"]["train"]["history"].float()
    y=c["splits"]["train"]["target"].float()
    mh=n.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=n.std.view(1,1,-1,*([1]*(h.ndim-3)))
    return ((h-mh)/sh).contiguous(),((y-mh)/sh).contiguous()
TRAIN={t:normalized_train(t) for t in TASKS}

def batch_plan(n,b,seed,steps):
    rng=np.random.default_rng(seed); out=[]
    while len(out)<steps:
        perm=rng.permutation(n); usable=(len(perm)//b)*b
        for i in range(0,usable,b):
            out.append(perm[i:i+b].astype(np.int64))
            if len(out)>=steps: break
    return out

def eval_val(model,loader,task):
    model.eval(); n=NORMS[task]; ms=[]
    with torch.no_grad():
        for h,y_n in loader:
            h=h.to(DEVICE); y_n=y_n.to(DEVICE)
            p_n=model.rollout(h,TRAIN_HORIZON)
            shp=[1,1,-1]+[1]*(y_n.ndim-3)
            mean=n.mean.view(*shp).to(DEVICE); std=n.std.view(*shp).to(DEVICE)
            p=p_n*std+mean; y=y_n*std+mean
            ms.append(float(((p-y)**2).mean()))
    return math.sqrt(np.mean(ms))

def train_one(model,task,va,seed,run_id,cp):
    model=model.to(DEVICE).float()
    lr=LR[run_id.split("__")[1]]
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=MAX_UPDATES,eta_min=lr*0.1)
    th,ty=TRAIN[task]; b=BATCH_1D if TASKS[task]["dim"]==1 else BATCH_2D
    bp=batch_plan(len(th),b,seed,MAX_UPDATES)
    start=0; best=float("inf"); best_step=0; best_sd=None; cref=float("inf"); bad=0; hist=[]; elapsed0=0.
    if cp.exists():
        ck=torch.load(cp,map_location="cpu",weights_only=False)
        load_portable_state_dict(model,ck["model"]); opt.load_state_dict(ck["optimizer"]); sch.load_state_dict(ck["scheduler"])
        start=int(ck["step"]); best=float(ck["best"]); best_step=int(ck["best_step"]); best_sd=ck.get("best_model")
        cref=float(ck.get("cref",best)); bad=int(ck.get("bad",0)); hist=list(ck.get("history",[])); elapsed0=float(ck.get("elapsed",0.))
        print("[resume]",run_id,start)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0=time.perf_counter(); reason="MAX_UPDATES"
    for s in range(start+1,MAX_UPDATES+1):
        idx=torch.as_tensor(bp[s-1],dtype=torch.long)
        h=th.index_select(0,idx).to(DEVICE); y=ty.index_select(0,idx).to(DEVICE)
        model.train(); opt.zero_grad(set_to_none=True)
        p=model.rollout(h,TRAIN_HORIZON); loss=F.mse_loss(p,y)
        if not torch.isfinite(loss): raise FloatingPointError((run_id,s))
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sch.step()
        stop=False
        if s==1 or s%EVAL_EVERY==0 or s==MAX_UPDATES:
            vr=eval_val(model,va,task)
            if vr<best: best=vr; best_step=s; best_sd=portable_state_dict(model)
            if not math.isfinite(cref): cref=vr; bad=0
            elif vr<cref*(1-REL_MIN_DELTA): cref=vr; bad=0
            elif s>=MIN_UPDATES: bad+=1
            hist.append({"step":s,"train_mse":float(loss.detach()),"val_rmse":vr,"best":best,"best_step":best_step,"bad":bad})
            print(f"[{run_id}] {s}/{MAX_UPDATES} val={vr:.6g} best={best:.6g}@{best_step}")
            if s>=MIN_UPDATES and bad>=PATIENCE_EVALS: stop=True; reason="CONVERGED_PATIENCE"
        if s%100==0 or s==MAX_UPDATES or stop:
            torch.cuda.synchronize(); elapsed=elapsed0+(time.perf_counter()-t0)
            torch.save({"step":s,"model":portable_state_dict(model),"optimizer":opt.state_dict(),"scheduler":sch.state_dict(),
                        "best":best,"best_step":best_step,"best_model":best_sd,"cref":cref,"bad":bad,"history":hist,"elapsed":elapsed},cp)
            if s%HF_CHECKPOINT_EVERY==0 or s==MAX_UPDATES or stop: sync_ckpt(cp,run_id,s)
        if stop: break
    torch.cuda.synchronize(); elapsed=elapsed0+(time.perf_counter()-t0); peak=int(torch.cuda.max_memory_allocated())
    if best_sd is None: best_sd=portable_state_dict(model)
    load_portable_state_dict(model,best_sd)
    return model,{"best_val_rmse":best,"best_val_step":best_step,"actual_updates":hist[-1]["step"],"stop_reason":reason,
                  "train_seconds":elapsed,"peak_vram_bytes":peak,"history":hist}

for seed in SEEDS:
    for task in TASKS:
        _,va=loaders(task,seed)
        hist_raw=LONG[task]["history"]
        target_raw=LONG[task]["target"]
        horizons=LONG[task]["available_horizons"]
        eval_h=[h for h in horizons if h in REQUESTED_HORIZONS]
        for name in MODELS:
            run_id=f"A4R1__{name}__{task}__s{seed}"
            rp=OUT/f"{run_id}_result.json"; hp=OUT/f"{run_id}_history.csv"; cp=CKPT/f"{run_id}_latest.pt"
            if rp.exists():
                try: old=json.loads(rp.read_text())
                except: old={}
                if old.get("status")=="COMPLETED":
                    print("[skip]",run_id); continue
            seed_all(seed); m=build_model(task,name)
            try:
                m,ti=train_one(m,task,va,seed,run_id,cp)
                pd.DataFrame(ti["history"]).to_csv(hp,index=False)
                clean=long_eval(m,task,hist_raw,target_raw,eval_h,rng_seed=seed)
                Hrob=24
                noise={}
                for nf in [0.01,0.05,0.10]:
                    noise[str(nf)]=long_eval(m,task,hist_raw,target_raw,[Hrob],noise_frac=nf,rng_seed=seed+1234)[Hrob]
                histloss={}
                for mode in ["drop_oldest_1","drop_oldest_2","last_frame_only"]:
                    histloss[mode]=long_eval(m,task,hist_raw,target_raw,[Hrob],history_mode=mode,rng_seed=seed)[Hrob]
                row={
                    "status":"COMPLETED","model":name,"task":task,"seed":seed,"params":count_params(m),
                    **{k:v for k,v in ti.items() if k!="history"},
                    "clean_long":clean,"noise_24":noise,"history_loss_24":histloss,
                }
                save_json(row,rp)
            except Exception as e:
                row={"status":"FAILED","model":name,"task":task,"seed":seed,"error":repr(e)}
                save_json(row,rp)
            sync_done(run_id,rp,hp,cp)
            if row.get("status")=="COMPLETED": cp.unlink(missing_ok=True)
            del m; torch.cuda.empty_cache(); gc.collect()

rows=[]
for p in OUT.glob("A4R1__*_result.json"):
    try: rows.append(json.loads(p.read_text()))
    except: pass
run_df=pd.DataFrame(rows)
run_df.to_csv(OUT/"A4_R1_run_results.csv",index=False)
if len(run_df)!=48 or not (run_df.status=="COMPLETED").all():
    raise RuntimeError("A4-R1 incomplete; restart and HF resume.")


## 8. Analysis, figures, gate

In [ ]:

# ============================================================
# 8. Analysis
# ============================================================
long_rows=[]; noise_rows=[]; hist_rows=[]
for _,r in run_df.iterrows():
    for h,v in r["clean_long"].items():
        long_rows.append({"model":r.model,"task":r.task,"seed":r.seed,"horizon":int(h),
                          "rmse":v["rmse"],"relative_l2":v["relative_l2"]})
    base24=r["clean_long"].get("24",r["clean_long"].get(24))
    for nf,v in r["noise_24"].items():
        noise_rows.append({"model":r.model,"task":r.task,"seed":r.seed,"noise_frac":float(nf),
                           "rmse":v["rmse"],"clean_rmse":base24["rmse"],
                           "degradation_pct":100*(v["rmse"]-base24["rmse"])/base24["rmse"]})
    for mode,v in r["history_loss_24"].items():
        hist_rows.append({"model":r.model,"task":r.task,"seed":r.seed,"mode":mode,
                          "rmse":v["rmse"],"clean_rmse":base24["rmse"],
                          "degradation_pct":100*(v["rmse"]-base24["rmse"])/base24["rmse"]})

long_df=pd.DataFrame(long_rows); noise_df=pd.DataFrame(noise_rows); hist_df=pd.DataFrame(hist_rows)
long_df.to_csv(OUT/"A4_R1_long_rollout_seedwise.csv",index=False)
noise_df.to_csv(OUT/"A4_R1_noise_robustness.csv",index=False)
hist_df.to_csv(OUT/"A4_R1_history_loss_robustness.csv",index=False)

summary=long_df.groupby(["model","task","horizon"]).agg(
    rmse_mean=("rmse","mean"),rmse_std=("rmse","std"),rel_mean=("relative_l2","mean")
).reset_index()
summary.to_csv(OUT/"A4_R1_long_rollout_summary.csv",index=False)

# growth relative to h=6 per model/task
stab=[]
for (m,t),g in summary.groupby(["model","task"]):
    g=g.sort_values("horizon")
    r6=float(g[g.horizon==6].rmse_mean.iloc[0])
    hmax=int(g.horizon.max()); rmax=float(g[g.horizon==hmax].rmse_mean.iloc[0])
    stab.append({"model":m,"task":t,"hmax":hmax,"rmse6":r6,"rmse_hmax":rmax,"growth_ratio":rmax/r6})
stab_df=pd.DataFrame(stab)
rank=stab_df.groupby("model").agg(
    mean_growth_ratio=("growth_ratio","mean"),
    median_growth_ratio=("growth_ratio","median"),
).reset_index().sort_values("mean_growth_ratio")
rank.to_csv(OUT/"A4_R1_stability_ranking.csv",index=False)

# figures
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white",
    "font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
    "mathtext.fontset":"stix","font.size":9,"axes.linewidth":0.8,
    "xtick.direction":"in","ytick.direction":"in","legend.frameon":False,
    "pdf.fonttype":42,"ps.fonttype":42,
})
LABEL={"CIDM-Foundation-v1":"CIDM-Core","CIDM-Complete":"CIDM-Complete",
       "PDEBench-FNO":"PDEBench FNO","NeuralOperator-FNO":"NeuralOperator FNO"}
def saveall(fig,stem):
    for ext in ["pdf","svg","png"]:
        kw={"bbox_inches":"tight"}
        if ext=="png": kw["dpi"]=600
        fig.savefig(FIG/f"{stem}.{ext}",**kw)
    plt.close(fig)

# Error growth curves, normalized to h=6 per task/model
fig,axes=plt.subplots(2,2,figsize=(7.8,5.8))
for ax,task in zip(axes.ravel(),TASKS):
    for m in MODELS:
        g=summary[(summary.task==task)&(summary.model==m)].sort_values("horizon")
        base=float(g[g.horizon==6].rmse_mean.iloc[0])
        ax.plot(g.horizon,g.rmse_mean/base,marker="o",label=LABEL[m])
    ax.set_title(task); ax.set_xlabel("Rollout horizon"); ax.set_ylabel("RMSE / RMSE@6")
    ax.set_yscale("log")
axes[0,0].legend(fontsize=7)
fig.tight_layout(); saveall(fig,"Fig_A4_01_error_growth_curves")

# Long horizon task-normalized error
hcommon=max(COMMON_HORIZONS)
q=summary[summary.horizon==hcommon].copy()
mins=q.groupby("task").rmse_mean.min().to_dict()
q["normalized"]=q.apply(lambda r:r.rmse_mean/mins[r.task],axis=1)
overall=q.groupby("model").normalized.mean().reset_index()
fig,ax=plt.subplots(figsize=(4.5,3.0))
ax.bar(range(len(overall)),overall.normalized,edgecolor="black",linewidth=.4)
ax.set_xticks(range(len(overall))); ax.set_xticklabels([LABEL[x] for x in overall.model],rotation=30,ha="right")
ax.set_ylabel(f"Task-normalized RMSE @ {hcommon} steps")
fig.tight_layout(); saveall(fig,"Fig_A4_02_long_horizon_normalized_error")

# Noise robustness
ng=noise_df.groupby(["model","noise_frac"]).degradation_pct.mean().reset_index()
fig,ax=plt.subplots(figsize=(4.8,3.0))
for m in MODELS:
    g=ng[ng.model==m].sort_values("noise_frac")
    ax.plot(100*g.noise_frac,g.degradation_pct,marker="o",label=LABEL[m])
ax.set_xlabel("Input noise (% training-channel std)"); ax.set_ylabel("24-step RMSE degradation (%)")
ax.legend(fontsize=7); fig.tight_layout(); saveall(fig,"Fig_A4_03_noise_robustness")

# History loss
hg=hist_df.groupby(["model","mode"]).degradation_pct.mean().reset_index()
modes=["drop_oldest_1","drop_oldest_2","last_frame_only"]
fig,ax=plt.subplots(figsize=(6.0,3.0))
x=np.arange(len(modes)); w=.18
for i,m in enumerate(MODELS):
    vals=[float(hg[(hg.model==m)&(hg["mode"]==mo)].degradation_pct.iloc[0]) for mo in modes]
    ax.bar(x+(i-1.5)*w,vals,w,label=LABEL[m],edgecolor="black",linewidth=.3)
ax.set_xticks(x); ax.set_xticklabels(modes,rotation=20); ax.set_ylabel("24-step RMSE degradation (%)")
ax.legend(ncol=2,fontsize=7); fig.tight_layout(); saveall(fig,"Fig_A4_04_history_loss_robustness")

# Stability-compute
train_cost=run_df.groupby("model").train_seconds.mean().reset_index()
pareto=rank.merge(train_cost,on="model")
fig,ax=plt.subplots(figsize=(4.4,3.0))
for _,r in pareto.iterrows():
    ax.scatter(r.train_seconds,r.mean_growth_ratio,s=45,edgecolor="black",linewidth=.4)
    ax.annotate(LABEL[r.model],(r.train_seconds,r.mean_growth_ratio),xytext=(4,3),textcoords="offset points",fontsize=7)
ax.set_xlabel("Mean train time / run (s)"); ax.set_ylabel("Mean long/6-step RMSE growth")
fig.tight_layout(); saveall(fig,"Fig_A4_05_stability_compute_pareto")

# gate
checks={
    "frozen_architectures":True,
    "official_sources_exact":all(x["exact"] for x in checkout),
    "parameter_contract":bool(((param_df.params>=475000)&(param_df.params<=525000)).all()),
    "preflight_16of16":len(pre_df)==16 and bool(pre_df["pass"].all()),
    "common_horizon_at_least_24":max(COMMON_HORIZONS)>=24,
    "formal_48of48":len(run_df)==48 and bool((run_df.status=="COMPLETED").all()),
    "long_summary_written":(OUT/"A4_R1_long_rollout_summary.csv").exists(),
    "noise_written":(OUT/"A4_R1_noise_robustness.csv").exists(),
    "history_loss_written":(OUT/"A4_R1_history_loss_robustness.csv").exists(),
    "figures_complete":all((FIG/f"{s}.{e}").exists() for s in [
        "Fig_A4_01_error_growth_curves","Fig_A4_02_long_horizon_normalized_error",
        "Fig_A4_03_noise_robustness","Fig_A4_04_history_loss_robustness",
        "Fig_A4_05_stability_compute_pareto"] for e in ["pdf","svg","png"]),
}
verdict="PASS_A4_R1_LONG_ROLLOUT_GENERALIZATION" if all(checks.values()) else "A4_R1_INCOMPLETE"
REPORT["gate"]=checks; REPORT["verdict"]=verdict
save_json(REPORT,OUT/"CIDM_Foundation_A4_R1_REPORT.json")
gate=OUT/"A4_R1_gate_report.txt"
gate.write_text("\n".join(["CIDM Foundation A4-R1","="*70,f"VERDICT: {verdict}",""]+
                          [f"[{'PASS' if v else 'FAIL'}] {k}" for k,v in checks.items()]),encoding="utf-8")
print(gate.read_text())

zip_path=OUT/"CIDM_Foundation_A4_R1_LongRollout_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p!=zip_path and "checkpoints" not in p.parts and "sources" not in p.parts:
            z.write(p,p.relative_to(OUT))
sha=sha256_file(zip_path)
(OUT/"CIDM_Foundation_A4_R1_LongRollout_RESULTS.zip.sha256").write_text(sha+"\n")

try:
    manifest(None,None,"FINAL_PACKAGE")
    retry(api.upload_folder,repo_id=HF_REPO,repo_type="dataset",folder_path=str(OUT),
          path_in_repo=HF_PREFIX,ignore_patterns=["checkpoints/*","sources/*"],
          commit_message="[A4-R1] final long-rollout/generalization package")
except Exception as e:
    print("[HF final warning]",e)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Auto-download unavailable:",e)
